# 08 Feed Forward Network 前馈神经网络

前面我们已经学完了 Self-Attention、Multi-Head Attention 和位置编码。

进入 Transformer Encoder 之前，还需要补一个重要组件：

```text
Feed Forward Network，简称 FFN。
```

在 Transformer Encoder 里，一层通常不是只有 Multi-Head Attention。

它还会接一个 FFN。

先给一句核心直觉：

```text
Attention 负责让 token 之间交流。
FFN 负责让每个 token 自己内部加工。
```

这一节先不写代码，只把 FFN 的作用、结构和形状讲清楚。

## 1. 为什么 Attention 后面还需要 FFN

Multi-Head Attention 已经让每个 token 从上下文里拿到了信息。

比如“她”这个 token，经过 Attention 后可能已经融合了“小红”的信息。

但拿到信息以后，还需要进一步加工。

这就像课堂讨论：

```text
Attention：我听别人说了什么。
FFN：我把听到的信息在自己脑子里整理、变换、提炼一下。
```

所以 Transformer 里通常是：

```text
先让 token 之间交流。
再让每个 token 自己做非线性加工。
```

这就是 FFN 存在的意义。

## 2. FFN 和前面学过的 MLP 有什么关系

FFN 本质上就是一个小型 MLP。

前面学习神经网络时，我们已经见过类似结构：

$$
\operatorname{Linear}_1\rightarrow\text{激活函数}\rightarrow\operatorname{Linear}_2
$$

Transformer 里的 FFN 也是这个思路。

常见结构是：

- 第一层 Linear：把维度从 $D$ 扩展到 $d_{\mathrm{ff}}$
- 激活函数：引入非线性
- 第二层 Linear：把维度从 $d_{\mathrm{ff}}$ 压回 $D$

所以 FFN 不是完全陌生的新东西。

它是 MLP 在 Transformer 每个 token 表示上的应用。

## 3. Transformer 里的 FFN 通常写成什么

对一个 token 向量 $\mathbf{x}$，FFN 常见形式是：

$$
\operatorname{FFN}(\mathbf{x})=\sigma(\mathbf{x}W_1+b_1)W_2+b_2
$$

其中：

- $W_1$ 是第一层 Linear 的权重。
- $b_1$ 是第一层 bias。
- $\sigma$ 是激活函数，比如 ReLU 或 GELU。
- $W_2$ 是第二层 Linear 的权重。
- $b_2$ 是第二层 bias。

如果用原始 Transformer 论文里常见的写法，激活函数可以是 ReLU：

$$
\operatorname{FFN}(\mathbf{x})=\max(0,\mathbf{x}W_1+b_1)W_2+b_2
$$

现代模型里也经常使用 GELU 等激活函数。

入门阶段先记住结构：

$$
\operatorname{Linear}_1\rightarrow\operatorname{Activation}\rightarrow\operatorname{Linear}_2
$$

## 4. 为什么中间要有激活函数

如果 FFN 只是两个 Linear 连在一起，但中间没有激活函数：

$$
\operatorname{Linear}_1\rightarrow\operatorname{Linear}_2
$$

那么两个线性变换合起来，本质上仍然可以看成一个线性变换。

也就是说，它的表达能力不会真正变复杂。

加上激活函数后，结构变成：

$$
\operatorname{Linear}_1\rightarrow\text{非线性激活}\rightarrow\operatorname{Linear}_2
$$

这样模型才能学习更复杂的特征变换。

这和我们前面学习 MLP 时的道理一样：

```text
没有非线性，很多层线性层叠起来仍然只是线性变换。
有了非线性，网络才有更强表达能力。
```

## 5. FFN 是 position-wise 的

Transformer 里的 FFN 经常叫：

```text
Position-wise Feed Forward Network
```

position-wise 的意思是：

```text
对每个位置单独应用同一套 FFN。
```

假设有 4 个 token：

$$
\mathbf{x}_1,\ \mathbf{x}_2,\ \mathbf{x}_3,\ \mathbf{x}_4
$$

FFN 会分别处理它们：

$$
\begin{aligned}
\operatorname{FFN}(\mathbf{x}_1)\\
\operatorname{FFN}(\mathbf{x}_2)\\
\operatorname{FFN}(\mathbf{x}_3)\\
\operatorname{FFN}(\mathbf{x}_4)
\end{aligned}
$$

但注意：这 4 个位置使用的是同一套 FFN 参数。

所以：

```text
每个位置单独加工。
所有位置共享同一套加工方法。
```

## 6. FFN 不负责 token 之间交流

这一点非常重要。

FFN 处理第 $i$ 个 token 时，只看第 $i$ 个 token 当前的向量。

它不会直接拿第 $i$ 个 token 去和第 $j$ 个 token 算关系。

token 之间的信息交流，是 Attention 做的。

FFN 做的是每个 token 内部的特征变换。

可以这样对比：

```text
Multi-Head Attention：横向交流，不同 token 之间传信息。
FFN：纵向加工，每个 token 自己内部做变换。
```

所以 Encoder Layer 里两者通常配合使用：

```text
先交流，再加工。
```

## 7. 用一个 token 看 FFN 的形状

先只看一个 token。

假设这个 token 的向量维度是：

$$
D=8
$$

FFN 中间维度设为：

$$
d_{\mathrm{ff}}=32
$$

那么第一层 Linear：

$$
(1\times8)\cdot(8\times32)\rightarrow1\times32
$$

激活函数不改变形状：

$$
1\times32\rightarrow1\times32
$$

第二层 Linear：

$$
(1\times32)\cdot(32\times8)\rightarrow1\times8
$$

所以一个 token 经过 FFN 后，形状路线是：

$$
1\times D\rightarrow1\times d_{\mathrm{ff}}\rightarrow1\times D
$$

## 8. 用一整句话看 FFN 的形状

现在看一句话。

假设有 4 个 token，每个 token 是 8 维：

$$
X:4\times8
$$

第一层 Linear 把每个 token 从 8 维扩展到 32 维：

$$
(4\times8)\cdot(8\times32)\rightarrow4\times32
$$

激活函数：

$$
4\times32\rightarrow4\times32
$$

第二层 Linear 把每个 token 从 32 维压回 8 维：

$$
(4\times32)\cdot(32\times8)\rightarrow4\times8
$$

所以一整句话经过 FFN 后：

$$
4\times8\rightarrow4\times32\rightarrow4\times8
$$

位置数量 4 没有变。

变的是每个位置内部向量的内容。

## 9. 加上 batch 后的形状

真实训练时通常是 batch 输入。

输入形状是：

$$
B\times N\times D
$$

FFN 的形状路线是：

$$
\begin{aligned}
B\times N\times D
&\rightarrow B\times N\times d_{\mathrm{ff}}\\
&\rightarrow B\times N\times D
\end{aligned}
$$

这里要注意：

- $B$ 不变：样本数量不变。
- $N$ 不变：token 位置数量不变。
- $D$ 先扩展到 $d_{\mathrm{ff}}$，再压回 $D$。

这就是 Transformer FFN 最常见的形状主线。

## 10. 为什么中间维度 $d_{\mathrm{ff}}$ 往往更大

Transformer 里的 FFN 通常会先把维度扩展到更大。

比如原始 Transformer 中常见设置是：

$$
\begin{aligned}
d_{\mathrm{model}}&=512\\
d_{\mathrm{ff}}&=2048
\end{aligned}
$$

也就是中间维度大约是模型维度的 4 倍。

为什么要这样？

可以先这样理解：

```text
先把向量展开到更大的特征空间。
在更大的空间里做非线性加工。
再压回原来的模型维度。
```

这有点像做题时先把思路展开，充分加工后再整理成最终答案。

中间维度更大，给了模型更多空间去组合、筛选和变换特征。

## 11. 为什么最后要压回 $D$

既然中间维度更大，为什么不一直保持 $d_{\mathrm{ff}}$ 呢？

因为 Transformer 的各层通常希望输入输出维度一致。

比如 Encoder Layer 前后都保持：

$$
B\times N\times D
$$

这样后面才能继续接下一层 Multi-Head Attention、残差连接和 LayerNorm。

如果 FFN 输出变成：

$$
B\times N\times d_{\mathrm{ff}}
$$

后面的结构就要跟着改变，残差连接也会遇到形状不一致的问题。

所以常见做法是：

```text
扩展是为了加工。
压回是为了继续保持统一维度。
```

## 12. FFN 和分类头不是一回事

因为 FFN 也是 Linear 结构，所以容易和分类头混淆。

但它们不是一回事。

分类头通常出现在模型最后，用来把隐藏表示变成类别分数或词表分数。

例如 MNIST 分类时，最后输出 10 个类别分数。

Transformer Encoder 里的 FFN 则是在每一层中间使用。

它不是直接输出最终答案。

它只是继续加工每个 token 的隐藏表示。

可以对比：

- FFN：中间层加工 token 表示，输出仍然是 $B\times N\times D$。
- 分类头：模型末尾生成任务需要的分数。

## 13. FFN 和 Attention 的分工

现在把 Attention 和 FFN 放在一起看。

Multi-Head Attention：

- 输入：$B\times N\times D$
- 作用：让不同 token 之间交换信息
- 输出：$B\times N\times D$

FFN：

- 输入：$B\times N\times D$
- 作用：对每个 token 单独做非线性特征加工
- 输出：$B\times N\times D$

这两个模块都保持形状不变，但改变信息内容。

区别是：

```text
Attention 改变信息来源：这个 token 从其他 token 那里拿信息。
FFN 改变内部表达：这个 token 对已有信息做进一步变换。
```

## 14. 为什么说 FFN 对所有位置共享参数

FFN 是 position-wise 的，但不是每个位置都有一套独立参数。

所有位置使用同一套：

$$
W_1,b_1,W_2,b_2
$$

例如：

```text
第 1 个 token 用这套 FFN。
第 2 个 token 也用这套 FFN。
第 3 个 token 也用这套 FFN。
```

这有点像 CNN 里卷积核在不同位置共享参数。

但注意，这里只是类比。

FFN 不是卷积。

类比只是帮助理解：

```text
同一套加工规则，被应用到不同位置。
```

## 15. FFN 会不会破坏位置信息

位置编码在进入 Transformer 前已经和 token embedding 相加。

经过 Attention 后，每个 token 表示里也可能已经融合了位置相关信息。

FFN 对每个位置的当前表示做加工。

它不会主动把不同位置混在一起。

但它可以对当前 token 表示里的内容信息、上下文信息、位置信息做非线性变换。

也就是说：

```text
位置之间的交流主要靠 Attention。
FFN 负责加工每个位置已经拥有的信息。
```

## 16. FFN 在 Encoder Layer 里的位置

一个 Transformer Encoder Layer 可以先粗略看成：

$$
\begin{aligned}
\text{输入}
&\rightarrow\operatorname{MultiHeadSelfAttention}\\
&\rightarrow\text{残差连接}+\operatorname{LayerNorm}\\
&\rightarrow\operatorname{FFN}\\
&\rightarrow\text{残差连接}+\operatorname{LayerNorm}\\
&\rightarrow\text{输出}
\end{aligned}
$$

这一节只学习 FFN。

下一节再专门补：

```text
残差连接
LayerNorm
```

等这两个也补完，再进入完整 Transformer Encoder，就不会一下子被多个新概念压住。

## 17. 常见误解 1：FFN 负责建模 token 之间关系

不对。

FFN 不直接建模 token 和 token 之间的关系。

它对每个位置单独应用。

token 之间的关系主要通过 Attention 建模。

FFN 只是对每个 token 当前拿到的信息做进一步加工。

所以不要把 FFN 看成第二个 Attention。

它更像是每个 token 自己的一段小型 MLP。

## 18. 常见误解 2：FFN 只是在增加参数

FFN 确实引入了参数。

但它不只是为了堆参数量。

它提供的是非线性特征变换能力。

Attention 负责混合上下文。

FFN 负责把混合后的表示进一步加工。

如果没有 FFN，Transformer 层的表达能力会弱很多。

可以这样理解：

```text
Attention 让信息流动起来。
FFN 让每个位置对流入的信息做更复杂的处理。
```

## 19. 常见误解 3：FFN 输出维度变回 $D$ 就说明没用

FFN 的输入和输出形状经常都是：

$$
B\times N\times D
$$

但形状一样不代表内容一样。

前面学 Attention 时我们已经见过这个现象。

Self-Attention 输入输出形状也可能一样，但 token 表示已经融合上下文。

FFN 也是如此。

它输出仍然是 $D$ 维，是为了保持模型结构统一。

但每个 token 的向量内容已经经过了非线性加工。

## 20. 本节小结

这一节先记住：

1. FFN 是 Transformer Encoder 里的重要组件。
2. FFN 本质上是应用在每个 token 上的小型 MLP。
3. 常见结构是 $\operatorname{Linear}_1\rightarrow\operatorname{Activation}\rightarrow\operatorname{Linear}_2$。
4. FFN 是 position-wise 的：每个位置单独处理，但共享同一套参数。
5. FFN 不负责 token 之间交流，token 之间交流主要靠 Attention。
6. FFN 的常见形状路线是 $B\times N\times D\rightarrow B\times N\times d_{\mathrm{ff}}\rightarrow B\times N\times D$。
7. 中间维度 $d_{\mathrm{ff}}$ 通常比 $D$ 大，是为了给非线性加工更多空间。
8. 最后压回 $D$ 是为了保持 Encoder Layer 输入输出维度一致。
9. FFN 不是分类头，它是中间层里的表示加工模块。
10. 进入完整 Transformer Encoder 前，还需要补残差连接和 LayerNorm。

## 21. 自测问题

1. 为什么 Attention 后面还需要 FFN？
2. FFN 和前面学过的 MLP 有什么关系？
3. Transformer 里的 FFN 常见结构是什么？
4. 为什么 FFN 中间需要激活函数？
5. position-wise FFN 是什么意思？
6. FFN 会不会直接让 token 之间交流？为什么？
7. 如果输入是 $B\times N\times D$，FFN 的常见形状路线是什么？
8. 为什么 $d_{\mathrm{ff}}$ 往往比 $D$ 大？
9. 为什么第二层 Linear 要把维度压回 $D$？
10. FFN 和分类头有什么区别？
11. 为什么说 FFN 对所有位置共享参数？
12. FFN 在 Transformer Encoder Layer 里通常位于哪个位置？